# Módulo 2 — Captura de Tráfego de Rede com NFStream
**Objetivo:** Capturar fluxos de rede ao vivo da máquina local usando NFStream  
**Interface:** enp3s0 (Ethernet)  
**Autor:** Marcel Teixeira Chiarelo  
**Data:** julho/2026

In [1]:
from nfstream import NFStreamer
import pandas as pd

print("Iniciando captura... aguarde 60 segundos.")

streamer = NFStreamer(
    source="enp3s0",
    statistical_analysis=True,
    idle_timeout=10,
    active_timeout=60
)

df = streamer.to_pandas()
print(f"Total de fluxos capturados: {len(df)}")
print(df.head())

Iniciando captura... aguarde 60 segundos.
Total de fluxos capturados: 158
   id  expiration_id          src_ip            src_mac   src_oui  src_port  \
0   0              0  192.168.100.50  7c:8a:e1:da:b0:f9  7c:8a:e1     56010   
1   1              0  192.168.100.50  7c:8a:e1:da:b0:f9  7c:8a:e1     45727   
2   2              0  192.168.100.50  7c:8a:e1:da:b0:f9  7c:8a:e1     59265   
3   3              0  192.168.100.50  7c:8a:e1:da:b0:f9  7c:8a:e1     56352   
4   4              0  192.168.100.50  7c:8a:e1:da:b0:f9  7c:8a:e1     60169   

            dst_ip            dst_mac   dst_oui  dst_port  ...  \
0     52.112.95.63  04:f3:52:3e:3b:7a  04:f3:52       443  ...   
1          8.8.8.8  04:f3:52:3e:3b:7a  04:f3:52        53  ...   
2  142.250.219.202  04:f3:52:3e:3b:7a  04:f3:52       443  ...   
3  142.251.157.119  04:f3:52:3e:3b:7a  04:f3:52       443  ...   
4          8.8.8.8  04:f3:52:3e:3b:7a  04:f3:52        53  ...   

   dst2src_fin_packets  application_name  application_

In [2]:
# Salvar em CSV
df.to_csv("captura_nfstream.csv", index=False)
print("Arquivo salvo: captura_nfstream.csv")
print(f"Dimensões: {df.shape}")
print(f"\nColunas disponíveis:\n{list(df.columns)}")

Arquivo salvo: captura_nfstream.csv
Dimensões: (158, 86)

Colunas disponíveis:
['id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui', 'src_port', 'dst_ip', 'dst_mac', 'dst_oui', 'dst_port', 'protocol', 'ip_version', 'vlan_id', 'tunnel_id', 'bidirectional_first_seen_ms', 'bidirectional_last_seen_ms', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_first_seen_ms', 'src2dst_last_seen_ms', 'src2dst_duration_ms', 'src2dst_packets', 'src2dst_bytes', 'dst2src_first_seen_ms', 'dst2src_last_seen_ms', 'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes', 'bidirectional_min_ps', 'bidirectional_mean_ps', 'bidirectional_stddev_ps', 'bidirectional_max_ps', 'src2dst_min_ps', 'src2dst_mean_ps', 'src2dst_stddev_ps', 'src2dst_max_ps', 'dst2src_min_ps', 'dst2src_mean_ps', 'dst2src_stddev_ps', 'dst2src_max_ps', 'bidirectional_min_piat_ms', 'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms', 'src2dst_min_piat_ms', 'src2dst_m

In [3]:
# Distribuição dos protocolos capturados
print("Protocolos mais frequentes:")
print(df["protocol"].value_counts().head(10))

Protocolos mais frequentes:
protocol
17    98
6     60
Name: count, dtype: int64


## Reflexão

Foram capturados 158 fluxos em 60 segundos de tráfego normal, gerado
principalmente pela navegação em uma guia anônima acessando google.com.

Os protocolos predominantes foram UDP (98 fluxos) e TCP (60 fluxos),
correspondendo respectivamente a consultas DNS e conexões HTTPS/TLS.
Aplicações identificadas incluem DNS.Google, QUIC.Google e TLS,
refletindo o tráfego típico de navegação web moderna.

O NFStream gerou 86 colunas de features, valor próximo às 78 features
do CIC-IDS2017. As colunas seguem nomenclatura diferente — por exemplo,
o CIC-IDS2017 usa "Flow Duration" enquanto o NFStream usa
"bidirectional_duration_ms" — mas representam os mesmos conceitos
estatísticos de fluxo de rede.

A principal diferença em relação ao CIC-IDS2017 é a ausência de rótulos:
o tráfego capturado é todo benigno e não possui coluna "Label". No dataset,
cada fluxo está rotulado com o tipo de ataque ou "BENIGN", o que só é
possível porque a captura foi feita em ambiente controlado com ataques
simulados em horários conhecidos.